# VQA Yes/No — Show, Attend and Tell (adapted)

Tests whether the paper's VGG encoder + soft attention transfers to binary VQA with minimal adaptation.

**Assumes:** COCO images already downloaded to Drive from a previous run.

**Steps:**
1. Setup (clone repo + install deps)
2. Mount Drive
3. Link images from Drive (no re-download)
4. Download VQA v2 annotations (if missing)
5. Build / load vocabulary
6. Train
7. Evaluate
8. Single-image inference

## 1. Setup

In [2]:
import os, sys

REPO = '/content/Neural_Image_Caption_Generator'

if not os.path.isdir(REPO):
    !git clone -b experiment/visualQA https://github.com/Miwi343/Neural_Image_Caption_Generator.git {REPO}
else:
    !git -C {REPO} pull

if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

!pip install -q -r requirements.txt

Already up to date.


In [4]:
import torch
print('torch :', torch.__version__)
print('cuda  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu   :', torch.cuda.get_device_name(0))
    print('mem   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

torch : 2.10.0+cu128
cuda  : True
gpu   : Tesla T4
mem   : 15.6 GB


## 2. Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os, shutil
from pathlib import Path

DRIVE_VQA = Path('/content/drive/MyDrive/vqa_results')
DRIVE_VQA.mkdir(parents=True, exist_ok=True)

def link_dir(local_name: str):
    local  = Path(local_name)
    remote = DRIVE_VQA / local_name
    remote.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        return
    if local.exists():
        if not any(local.iterdir()):
            shutil.rmtree(local)
        else:
            return
    os.symlink(remote, local)
    print(f'Linked: {local} -> {remote}')

link_dir('checkpoints_vqa')
link_dir('results_vqa')
print('Output dirs ready.')

Output dirs ready.


## 3. Link Images from Drive

Images are already on Drive from the previous run. This cell just symlinks them into the expected path — no re-downloading.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path('data/vqa')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_IMAGES = Path('/content/drive/MyDrive/vqa_results/images')
LOCAL_IMAGES = DATA_ROOT / 'images'

if not DRIVE_IMAGES.exists():
    raise RuntimeError(
        'Images not found at MyDrive/vqa_results/images/. '
        'Check your Drive path or re-run the original download notebook.'
    )

if LOCAL_IMAGES.is_symlink() or LOCAL_IMAGES.exists():
    print(f'Images already linked at {LOCAL_IMAGES}')
else:
    os.symlink(DRIVE_IMAGES, LOCAL_IMAGES)
    print(f'Linked: {LOCAL_IMAGES} -> {DRIVE_IMAGES}')

# Quick count to confirm
for split in ('train2014', 'val2014'):
    split_dir = DRIVE_IMAGES / split
    if split_dir.exists():
        n = len(list(split_dir.glob('*.jpg')))
        print(f'  {split}: {n:,} images')
    else:
        print(f'  {split}: NOT FOUND')

## 4. Download VQA v2 Annotations (if missing)

In [ ]:
import subprocess, zipfile
from pathlib import Path

DATA_ROOT = Path('data/vqa')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

ANNOTATIONS = [
    ('https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip',
     'v2_mscoco_train2014_annotations.json'),
    ('https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip',
     'v2_mscoco_val2014_annotations.json'),
    ('https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip',
     'v2_OpenEnded_mscoco_train2014_questions.json'),
    ('https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip',
     'v2_OpenEnded_mscoco_val2014_questions.json'),
]

for url, expected_json in ANNOTATIONS:
    dest = DATA_ROOT / expected_json
    if dest.exists():
        print(f'OK  {expected_json}')
        continue
    zip_dest = DATA_ROOT / url.split('/')[-1]
    print(f'Downloading {url.split("/")[-1]} ...')
    subprocess.run(['wget', '-q', '-O', str(zip_dest), url], check=True)
    with zipfile.ZipFile(zip_dest) as z:
        z.extractall(DATA_ROOT)
    zip_dest.unlink()
    print(f'  -> {expected_json}')

## 5. Build / Load Vocabulary

In [ ]:
from pathlib import Path
from vqa.dataset import QuestionVocabulary, build_and_save_vocab

DATA_ROOT  = 'data/vqa'
VOCAB_PATH = 'data/vqa/vocab.json'

if Path(VOCAB_PATH).exists():
    vocab = QuestionVocabulary.load(VOCAB_PATH)
    print(f'Loaded vocab: {len(vocab):,} tokens')
else:
    vocab = build_and_save_vocab(DATA_ROOT, VOCAB_PATH, max_size=15_000)

# Back up to Drive so it survives session resets
!mkdir -p "/content/drive/MyDrive/vqa_results/data/vqa"
!cp data/vqa/vocab.json "/content/drive/MyDrive/vqa_results/data/vqa/vocab.json"
print('Vocab backed up to Drive.')

## 5b. Pre-extract VGG Features (run once, ~30-60 min)

Runs the VGG encoder over every image once and saves `(196, 512)` feature tensors to Drive.
Subsequent training epochs load tensors instead of JPEGs — ~10x faster per epoch.
Skip this cell if `features_train2014.pt` already exists on Drive.

In [ ]:
# Extract features (resumes from Drive if partially done — backs up each shard immediately)
from vqa.extract_features import main as extract_main
extract_main()

In [ ]:
# Sanity check — confirm dataset sees samples and inspect class balance
from vqa.dataset import VQAYesNoDataset

for split in ('train', 'val'):
    ds = VQAYesNoDataset(DATA_ROOT, vocab, split=split)
    if len(ds) == 0:
        print(f'WARNING: 0 {split} samples — images missing?')
        continue
    yes = sum(p[2] for p in ds.pairs)
    print(f'{split:5s}: {len(ds):,} samples  |  yes {yes:,} ({yes/len(ds):.1%})  no {len(ds)-yes:,} ({1-yes/len(ds):.1%})')

print('\nSample questions (val):')
val_ds = VQAYesNoDataset(DATA_ROOT, vocab, split='val')
for _, q, label in val_ds.pairs[:5]:
    print(f'  {q!r:60s} -> {"yes" if label else "no"}')

## 6. Train

In [ ]:
# Runs with all defaults from vqa/train.py
# To override: train_main({'lr': 1e-3, 'dropout': 0.3, 'tag': 'exp1'})
from vqa.train import main as train_main
train_main()

In [ ]:
import csv, math, matplotlib.pyplot as plt
from pathlib import Path

log_path = Path('results_vqa/training_log.csv')
assert log_path.exists(), 'No training log found — run the training cell first.'

with open(log_path) as f:
    rows = list(csv.DictReader(f))

def _f(v):
    try:
        x = float(v)
        return None if math.isnan(x) else x
    except (ValueError, TypeError):
        return None

epochs     = [int(r['epoch'])           for r in rows]
train_acc  = [_f(r['train_acc'])        for r in rows]
val_acc    = [_f(r['val_acc'])          for r in rows]
train_loss = [_f(r['train_loss'])       for r in rows]
val_loss   = [_f(r['val_loss'])         for r in rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_acc, label='train')
ax1.plot(epochs, val_acc,   label='val')
ax1.set_title('Accuracy'); ax1.set_xlabel('epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, train_loss, label='train')
ax2.plot(epochs, val_loss,   label='val')
ax2.set_title('Loss'); ax2.set_xlabel('epoch'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results_vqa/training_curves.png', dpi=150)
plt.show()

## 7. Evaluate

In [ ]:
import torch, json
from pathlib import Path
from tqdm import tqdm
from vqa.dataset import QuestionVocabulary, get_vqa_dataloader
from vqa.model import VQAModel

CHECKPOINT = 'checkpoints_vqa/best.pt'
VOCAB_PATH  = 'data/vqa/vocab.json'
DATA_ROOT   = 'data/vqa'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab  = QuestionVocabulary.load(VOCAB_PATH)

ckpt  = torch.load(CHECKPOINT, map_location=device)
model = VQAModel(vocab_size=ckpt['vocab_size']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

loader = get_vqa_dataloader(DATA_ROOT, vocab, 'val', batch_size=64)

correct, total, yes_pred, yes_true = 0, 0, 0, 0
with torch.no_grad():
    for images, questions, q_lens, labels in tqdm(loader, desc='Evaluating'):
        images, questions, q_lens, labels = (
            images.to(device), questions.to(device), q_lens.to(device), labels.to(device)
        )
        logits, _ = model(images, questions, q_lens)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        yes_pred += preds.sum().item()
        yes_true += labels.sum().item()

results = {
    'val_accuracy':   correct / total,
    'yes_pred_rate':  yes_pred / total,
    'yes_true_rate':  yes_true / total,
    'total_samples':  total,
    'checkpoint_epoch': ckpt['epoch'],
}

print(f"Val accuracy  : {results['val_accuracy']:.4f}")
print(f"Yes pred rate : {results['yes_pred_rate']:.3f}  (true rate: {results['yes_true_rate']:.3f})")
print(f"Samples       : {total:,}")

out = Path('results_vqa/eval_results.json')
out.write_text(json.dumps(results, indent=2))
print(f'Saved -> {out}')

## 8. Single-Image Inference

Run **8a once** to load the model. Then re-run **8b** freely with different images/questions — no reload penalty.

In [ ]:
# 8a — load model once, keep in memory
import torch
from torchvision import transforms
from vqa.dataset import QuestionVocabulary, PAD_IDX
from vqa.model import VQAModel

CHECKPOINT = 'checkpoints_vqa/best.pt'
VOCAB_PATH  = 'data/vqa/vocab.json'
MAX_Q_LEN   = 20

_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_vocab  = QuestionVocabulary.load(VOCAB_PATH)

_ckpt  = torch.load(CHECKPOINT, map_location=_device)
_model = VQAModel(vocab_size=_ckpt['vocab_size']).to(_device)
_model.load_state_dict(_ckpt['model_state'])
_model.eval()

_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print(f"Model ready  — epoch {_ckpt['epoch']}, val acc {_ckpt['val_acc']:.4f}")

In [ ]:
# 8b — upload image + ask question (re-run as many times as you like)
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from google.colab import files

QUESTION = 'Is there a person in this image?'  # <-- change this

uploaded   = files.upload()
IMAGE_PATH = next(iter(uploaded))

img_pil = Image.open(IMAGE_PATH).convert('RGB')
img_t   = _transform(img_pil).unsqueeze(0).to(_device)

ids   = _vocab.encode(QUESTION, max_len=MAX_Q_LEN)
q_len = len(ids)
ids  += [PAD_IDX] * (MAX_Q_LEN - q_len)
q_t   = torch.tensor([ids], dtype=torch.long).to(_device)
ql_t  = torch.tensor([q_len], dtype=torch.long).to(_device)

with torch.no_grad():
    logit, alpha = _model(img_t, q_t, ql_t)

prob       = torch.sigmoid(logit).item()
answer     = 'yes' if prob >= 0.5 else 'no'
confidence = prob if answer == 'yes' else 1 - prob
alpha_np   = alpha.squeeze(0).cpu().numpy()

print(f'Q: {QUESTION}')
print(f'A: {answer.upper()}  (confidence: {confidence:.1%})')

# Attention map — mirrors paper Fig. 2/3
img_224     = np.array(img_pil.resize((224, 224)))
attn_smooth = gaussian_filter(np.kron(alpha_np.reshape(14, 14), np.ones((16, 16))), sigma=8)
attn_norm   = (attn_smooth - attn_smooth.min()) / (attn_smooth.max() - attn_smooth.min() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_224); axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(img_224)
axes[1].imshow(attn_norm, alpha=0.55, cmap='hot')
axes[1].set_title(f'A: {answer.upper()} ({confidence:.1%})'); axes[1].axis('off')
fig.suptitle(f'Q: {QUESTION}', fontsize=12)
plt.tight_layout()
plt.savefig('results_vqa/inference_attention.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. VGG Feature PCA Visualization

Runs any image through the frozen VGG encoder → `(196, 512)` spatial features over a 14×14 grid. PCA reduces 512 dims to 3, mapped directly to RGB. Regions with similar deep features get similar colors — sky, objects, and background separate by color with zero labels or training.

Run **9a once** to load the encoder. Re-run **9b** on any image.

In [6]:
# 9a — load encoder once
import torch
from torchvision import transforms
from models.encoder import Encoder

_pca_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_pca_encoder = Encoder(fine_tune=False).to(_pca_device)
_pca_encoder.eval()

_pca_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print(f'Encoder ready on {_pca_device}')

Encoder ready on cuda


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from scipy.ndimage import zoom
from IPython.display import display
import os

IMAGE_PATH = '/content/drive/MyDrive/vqa_results/images/ithaca-falls.jpg'

img_pil = Image.open(IMAGE_PATH).convert('RGB')
img_224 = np.array(img_pil.resize((224, 224)))
img_t   = _pca_transform(img_pil).unsqueeze(0).to(_pca_device)

with torch.no_grad():
    feats = _pca_encoder(img_t).squeeze(0).cpu().numpy()

pca        = PCA(n_components=3)
components = pca.fit_transform(feats)  # (196, 3)

for i in range(3):
    lo, hi = components[:, i].min(), components[:, i].max()
    components[:, i] = (components[:, i] - lo) / (hi - lo + 1e-8)

pca_map   = components.reshape(14, 14, 3)
pca_large = zoom(pca_map, (16, 16, 1), order=1)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('VGG Feature PCA — each color = distinct deep-feature region', fontsize=13)

axes[0, 0].imshow(img_224);   axes[0, 0].set_title('Original');        axes[0, 0].axis('off')
axes[0, 1].imshow(pca_large); axes[0, 1].set_title('PCA RGB map');     axes[0, 1].axis('off')
axes[0, 2].imshow(img_224);   axes[0, 2].imshow(pca_large, alpha=0.55)
axes[0, 2].set_title('Overlay (α=0.55)');                               axes[0, 2].axis('off')

for i, (label, cmap) in enumerate(zip(
    ['PC1 (most variance)', 'PC2', 'PC3'], ['Reds', 'Greens', 'Blues']
)):
    comp_large = zoom(components[:, i].reshape(14, 14), (16, 16), order=1)
    axes[1, i].imshow(img_224, alpha=0.4)
    axes[1, i].imshow(comp_large, cmap=cmap, alpha=0.7)
    axes[1, i].set_title(f'{label}  ({pca.explained_variance_ratio_[i]:.1%} var)')
    axes[1, i].axis('off')

plt.tight_layout()
os.makedirs('results_vqa', exist_ok=True)
out = 'results_vqa/pca_ithaca_falls.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
display(fig)
plt.close()
print(f'Saved → {out}')